## Imports

In [ ]:
import numpy as np
import xarray as xr
import copy
import src.utils
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cmocean
import os
import pathlib
import cartopy.crs as ccrs
import pandas as pd
import scipy.stats
import tqdm
import time
import xesmf as xe

## Set seaborn plotting style
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})
sns.set_palette("colorblind")

## initialize RNG
rng = np.random.default_rng(seed=100)

## Funcs

In [ ]:
## specify lons/lats for E/W boxes
LONS_W = [120, 200]
LONS_E = [200, 280]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-30, 0, 30],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[50, 120 - 160],
            # xlocs=[],
            ylocs=[-30, 0, 30],
        )
        gl_.top_labels = False
        # gl_.bottom_labels = True
    # gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return


import cartopy.crs as ccrs


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


def plot_pr_diff(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance_r",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_sst_sigma(ax, data, lev_min=0.3, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.amp",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_pr(ax, data, lev_min=1, dx=1, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.rain",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

## Data loading

## Load the data

### init. cluster

In [ ]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(n_workers=2)
client = Client(cluster)
client

### do loading

In [ ]:
SAVE_FP = pathlib.Path("/glade/work/kcarr/lme_data")
SAVE_FNAME = SAVE_FP / "sst_on_pr_grid.nc"

## load precip
sst_total = xr.open_dataarray(SAVE_FNAME).compute()

## compute mean and annual avg
sst_bar = sst_total.mean(["time", "member"])
sst = sst_total - sst_bar
sst_avg = sst.resample({"time": "YS"}).mean().mean(["member", "lat", "lon"])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(sst_avg)
src.utils.add_hticks([ax], yticks=[-1, 0, 0.5], ylines=[0])
plt.show()

Compute EOFs

In [ ]:
import xeofs as xe

## specs for EOFs
eofs_kwargs = dict(n_modes=300, standardize=False, use_coslat=True, center=True)

## initialize EOF model
eofs = xe.single.EOF(**eofs_kwargs)

## compute
eofs.fit(sst, dim=["time", "member"])

Ensure matrices are normalized properly

In [ ]:
N = int(len(sst.member) * len(sst.time))
Q = eofs.scores(normalized=True)
# Z = eofs.components(normalized=True)
Z = eofs.components(normalized=True)
sigma = np.sqrt(N * eofs.explained_variance())
W = src.utils.get_coslat_weights(Z)

## compute inner products
n_testmode = 10
idx = dict(mode=slice(None, n_testmode))
QtQ = xr.dot(
    Q.isel(idx), Q.isel(idx).rename({"mode": "mode_out"}), dims=["member", "time"]
)

ZtZ = xr.dot(
    Z.isel(idx).fillna(0),
    Z.isel(idx).fillna(0).rename({"mode": "mode_out"}),
    dims=["lat", "lon"],
)

## check they're identities
eye = np.eye(n_testmode)
print(np.allclose(ZtZ.values, eye))
print(np.allclose(QtQ.values, eye))

Ensure we can reconstruct the data

In [ ]:
## random sample
idx = dict(member=3, time=30)

## reconstruct using package and custom
gt = sst.isel(idx)
r0 = eofs.inverse_transform(eofs.scores().isel(idx))
r1 = xr.dot(Z / W, sigma, Q.isel(idx), dim="mode")

## check reconstructions the same
np.allclose(r0, r1, atol=1e-3, equal_nan=True)
print(f"{xr.corr(gt, r0).values.item():.6f}")
print(f"{xr.corr(gt, r1).values.item():.6f}")
print(f"{xr.corr(r0, r1).values.item():.6f}")

Same, but with custom cos-lat weighting

In [ ]:
src.utils.get_coslat_weights(sst)

In [ ]:
import xeofs as xe

## specs for EOFs
eofs_kwargs = dict(n_modes=300, standardize=False, use_coslat=False, center=True)

## initialize EOF model
eofs = xe.single.EOF(**eofs_kwargs)

## weight sst by coslat
X = sst / src.utils.get_coslat_weights(sst)

## compute
eofs.fit(sst, dim=["time", "member"])

Ensure matrices are normalized properly

In [ ]:
N = int(len(sst.member) * len(sst.time))
Q = eofs.scores(normalized=True)
# Z = eofs.components(normalized=True)
Z = eofs.components(normalized=True)
sigma = np.sqrt(N * eofs.explained_variance())
# W = src.utils.get_coslat_weights(Z)
W = 1

## compute inner products
n_testmode = 10
idx = dict(mode=slice(None, n_testmode))
QtQ = xr.dot(
    Q.isel(idx), Q.isel(idx).rename({"mode": "mode_out"}), dims=["member", "time"]
)

ZtZ = xr.dot(
    Z.isel(idx).fillna(0),
    Z.isel(idx).fillna(0).rename({"mode": "mode_out"}),
    dims=["lat", "lon"],
)

## check they're identities
eye = np.eye(n_testmode)
print(np.allclose(ZtZ.values, eye))
print(np.allclose(QtQ.values, eye))

Ensure we can reconstruct the data

In [ ]:
## random sample
idx = dict(member=3, time=30)

## reconstruct using package and custom
gt = sst.isel(idx)
r0 = eofs.inverse_transform(eofs.scores().isel(idx))
r1 = xr.dot(Z / W, sigma, Q.isel(idx), dim="mode")

## check reconstructions the same
np.allclose(r0, r1, atol=1e-3, equal_nan=True)
print(f"{xr.corr(gt, r0).values.item():.6f}")
print(f"{xr.corr(gt, r1).values.item():.6f}")
print(f"{xr.corr(r0, r1).values.item():.6f}")